# Export scikit-learn model → ONNX

Run this **after** `train_classifier.ipynb` has produced `model.pkl` and `vectorizer.pkl`.

Output: `public/model.onnx` and `public/vocab.json` — both go into your Svelte `public/` folder so Vite serves them as static assets.

### pip install skl2onnx onnx onnxruntime scikit-learn joblib

## 1. Load the trained pipeline

In [4]:
import joblib, json, re, numpy as np
from pathlib import Path

clf        = joblib.load('model.pkl')
vectorizer = joblib.load('vectorizer.pkl')

print(f'Model:      {type(clf).__name__}')
print(f'Vocab size: {len(vectorizer.vocabulary_)}')

Model:      LinearSVC
Vocab size: 30000


## 2. Ensure the classifier has predict_proba (calibrate LinearSVC if needed)

In [6]:
import re as _re, email as _email
from pathlib import Path
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split

def _clean(text):
    text = text.lower()
    text = _re.sub(r'http\S+|www\.\S+', ' url ', text)
    text = _re.sub(r'\S+@\S+', ' email ', text)
    text = _re.sub(r'\d+', ' num ', text)
    text = _re.sub(r'[^\w\s]', ' ', text)
    return _re.sub(r'\s+', ' ', text).strip()

def _parse(path):
    try:
        msg = _email.message_from_bytes(path.read_bytes())
        parts = []
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == 'text/plain':
                    try: parts.append(part.get_payload(decode=True).decode('utf-8', errors='ignore'))
                    except: pass
        else:
            try: parts.append(msg.get_payload(decode=True).decode('utf-8', errors='ignore'))
            except: parts.append(str(msg.get_payload()))
        return _clean(f"{msg.get('Subject','')} {' '.join(parts)}")
    except:
        return ''

if isinstance(clf, LinearSVC):
    print('LinearSVC detected — needs calibration for probability output.')
    print('Re-loading SpamAssassin corpus to fit calibrator...')

    DATA_DIR = Path('spamassassin')
    records = []
    for label in ('ham', 'spam'):
        for f in (DATA_DIR / label).rglob('*'):
            if f.is_file() and not f.suffix == '.bz2':
                t = _parse(f)
                if len(t) > 20:
                    records.append((t, 1 if label == 'spam' else 0))

    texts  = [r[0] for r in records]
    labels = [r[1] for r in records]
    X_tr, _, y_tr, _ = train_test_split(texts, labels, test_size=0.2,
                                         random_state=42, stratify=labels)
    X_tr_vec = vectorizer.transform(X_tr)

    calibrated = CalibratedClassifierCV(clf)
    calibrated.fit(X_tr_vec, y_tr)
    clf = calibrated
    print(f'Calibration done. Loaded {len(X_tr)} training emails.')
else:
    print(f'{type(clf).__name__} already supports predict_proba — no calibration needed.')

# Smoke-test
sample_text = 'Get a FREE prize NOW click here limited offer'
x_sample = vectorizer.transform([_clean(sample_text)])
pred  = clf.predict(x_sample)
proba = clf.predict_proba(x_sample)
print(f'Smoke-test → label={pred[0]}  proba={proba[0]}')


LinearSVC detected — needs calibration for probability output.
Re-loading SpamAssassin corpus to fit calibrator...
Calibration done. Loaded 4625 training emails.
Smoke-test → label=1  proba=[3.48957460e-04 9.99651043e-01]


## 3. Export vocabulary + IDF weights as JSON

ONNX supports TF-IDF → classifier pipelines **but** the TfidfVectorizer ONNX op
requires string input. It is cleaner — and more portable to `onnxruntime-web` —
to export only the **classifier** to ONNX and re-implement the TF-IDF transform
in JavaScript using the vocabulary + IDF weights we dump here.

In [10]:
BASE_DIR = Path.cwd().parent  # notebooks → project root
OUT = BASE_DIR / "svelte-app" / "public"

OUT.mkdir(parents=True, exist_ok=True)

vocab   = vectorizer.vocabulary_          # token → column index
idf     = vectorizer.idf_.tolist()        # list of float (one per feature)
ngrange = list(vectorizer.ngram_range)    # e.g. [1, 2]
# stop    = list(vectorizer.stop_words_) if vectorizer.stop_words_ else []
stop = list(getattr(vectorizer, 'stop_words_', None) or [])

# vocab_payload = {
#     'vocab':       vocab,
#     'idf':         idf,
#     'ngram_range': ngrange,
#     'sublinear_tf': vectorizer.sublinear_tf,
# }
vocab_payload = {
    'vocab':        {k: int(v) for k, v in vectorizer.vocabulary_.items()},  # fix here
    'idf':          idf,
    'ngram_range':  ngrange,
    'sublinear_tf': vectorizer.sublinear_tf,
}

vocab_path = OUT / 'vocab.json'
vocab_path.write_text(json.dumps(vocab_payload))
print(f'Wrote {vocab_path}  ({vocab_path.stat().st_size/1024:.0f} KB)')

Wrote /home/nyxox/Documents/Projects/emailClassifierSchool/svelte-app/public/vocab.json  (1159 KB)


## 4. Export the classifier to ONNX

In [11]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

n_features = len(vectorizer.vocabulary_)

# Export just the classifier (not the vectorizer)
# For CalibratedClassifierCV we must pass the inner estimator type to options
clf_type = type(clf)
initial_type = [('float_input', FloatTensorType([None, n_features]))]
onnx_model = convert_sklearn(
    clf,
    initial_types=initial_type,
    options={clf_type: {'zipmap': False}}
)

onnx_path = OUT / 'model.onnx'
onnx_path.write_bytes(onnx_model.SerializeToString())
print(f'Wrote {onnx_path}  ({onnx_path.stat().st_size/1024:.0f} KB)')

# Verify round-trip with onnxruntime
sess  = rt.InferenceSession(str(onnx_path))
x_vec = vectorizer.transform([_clean(sample_text)]).toarray().astype(np.float32)
out   = sess.run(None, {'float_input': x_vec})
label_out, proba_out = out[0], out[1]
print(f'ONNX prediction: label={label_out[0]}  proba={proba_out[0]}')


Wrote /home/nyxox/Documents/Projects/emailClassifierSchool/svelte-app/public/model.onnx  (1469 KB)
ONNX prediction: label=1  proba=[3.4897326e-04 9.9965107e-01]


## 5. Check output sizes

In [12]:
for f in OUT.iterdir():
    print(f'{f.name:30s}  {f.stat().st_size/1024:>8.1f} KB')

print()
print('Next step: cd svelte-app && npm install && npm run dev')

model.onnx                        1469.1 KB
vocab.json                        1158.9 KB

Next step: cd svelte-app && npm install && npm run dev
